# repark torture-dataset tour

Five generated families, read back through the repark facade, each stopping on the one
behavior it exists to expose:

| Family | What it tortures |
|---|---|
| `nested` | deep struct/list nesting with **capitalized** field names (`Legs`) |
| `schema_inference` | CSV inference conflicts the sampling budget can miss |
| `extreme_types` | decimal128 scale, digits beyond the 38-digit cap, UUID / paragraph / HTML |
| `secrets` | credential-named columns carrying obviously-synthetic values |
| `smartcsv` | BOM, preamble, ragged rows, a delimiter zoo, null tokens, bool spellings |

**Execution gating arrives with the examples-harness workstream.** No gate runs this
notebook today: it is executed by hand before it lands and committed with its outputs
cleared, so the diff stays reviewable.

**Where the data goes.** Every generator writes under the per-user cache root
(`$XDG_CACHE_HOME/repark-datasets/<family>`, else `~/.cache/...`) and refuses to write
inside a checkout. Nothing here lands in the repository.

**Scale.** This tour runs 2 000 rows at seed 42 so it finishes in minutes. The
generators' CLI default is 1 000 000 rows — pass `--rows` for a bigger local corpus:

```
python python/repark-parity/datasets/nested/datagen.py --rows 1000000 --seed 42
```

**Requires** the built native module (`make develop` or `make py-test-facade`).

In [ ]:
import importlib
import sys
import types
from pathlib import Path


def repark_repo_root() -> Path:
    """The checkout this notebook lives in (works from any cwd inside it)."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "AGENTS.md").is_file() and (candidate / "Cargo.toml").is_file():
            return candidate
    raise RuntimeError("run this notebook from inside a repark checkout")


REPO_ROOT = repark_repo_root()
DATASETS_DIR = REPO_ROOT / "python" / "repark-parity" / "datasets"

# The datasets tree is not a packaged module — load it the way the tests do.
if "repark_datasets" not in sys.modules:
    package = types.ModuleType("repark_datasets")
    package.__path__ = [str(DATASETS_DIR)]
    sys.modules["repark_datasets"] = package


def datagen(family: str) -> types.ModuleType:
    """Import one family's generator module."""
    return importlib.import_module(f"repark_datasets.{family}.datagen")


ROWS = 2_000  # tour scale; the CLI default is 1_000_000
SEED = 42

cache = importlib.import_module("repark_datasets._cache")
print("families:  ", ", ".join(cache.KNOWN_FAMILIES))
print("cache root:", cache.default_datasets_root())

In [ ]:
from repark import ReparkSession
from repark import functions as F  # noqa: N812

spark = ReparkSession.builder.appName("datasets-tour").getOrCreate()
print("repark", spark.version)

## 1. `nested` — capitalized `Legs`, seven levels deep

The signature behavior: a **string-form** `F.explode("Legs")` binds the mixed-case field
(unquoted identifiers fold to lowercase in the engine, so this used to fail), and
`dynamicFlatten` walks structs with parent-path prefixes while dropping the
`array<void>` column instead of exploding it.

In [ ]:
nested = datagen("nested")
nested_dir = nested.write_files(rows=ROWS, seed=SEED)  # cache root; never the checkout
print(nested_dir, "→", sorted(path.name for path in nested_dir.iterdir()))

frame = spark.read.parquet(str(nested_dir / "data.parquet"))
print(frame.columns)
frame.printSchema()

In [ ]:
# String-form explode of a CAPITALIZED list column (value AND Arrow type).
legs = frame.select(frame.id, F.explode("Legs").alias("leg"))
table = legs.to_arrow()
print(table.num_rows, "legs from", ROWS, "rows")
print(table.schema.field("leg").type)
print(table.to_pylist()[0])

In [ ]:
# dynamicFlatten: struct fields become {parent}_{field}; the inner list stays put.
flat_legs = legs.dynamicFlatten(explode_lists=False)
print(flat_legs.columns)
print(flat_legs.orderBy("id", "leg_leg_id").to_arrow().to_pylist()[0])

# Full depth: every list explodes, and the null-typed list is dropped, not exploded.
deep = frame.dynamicFlatten()
print(deep.columns)

# Counted through the export path on purpose: `deep.count()` currently reds inside the
# `push_down_leaf_projections` optimizer rule on this plan (reported, and pinned in
# test_datasets_facade.py) while `to_arrow()` returns the right rows.
print("rows after the full explode cartesian:", deep.to_arrow().num_rows)

## 2. `schema_inference` — the conflict the sample never sees

`smartCsv` infers from at most `samplingRows` data rows (10 000 by default) and always
materialises the whole file. This family puts an int32 → int64 widening (and a
string → float flip) at a configurable row, so the miss is reproducible on purpose.

`describe_ingest()` is the diagnostic surface: no inference decision is silent.

In [ ]:
inference = datagen("schema_inference")
inference_dir = inference.write_files(rows=ROWS, seed=SEED, conflict_at=ROWS // 2)
csv_path = inference_dir / "data.csv"

full = spark.read.smartCsv(str(csv_path), sep=",")
full_report = full.describe_ingest()
print("rows:", full_report["data_row_count"], "| scanned:", full_report["inference_rows_scanned"])
for column in full_report["columns"]:
    print(f"{column['name']:>16} → {column['resolved_type']:<20} nulls={column['null_count']}")

In [ ]:
# Cap the budget below the conflict row: the widening is never sampled.
capped = spark.read.smartCsv(str(csv_path), sep=",", samplingRows=64)
capped_report = capped.describe_ingest()
full_types = {column["name"]: column["resolved_type"] for column in full_report["columns"]}
capped_types = {column["name"]: column["resolved_type"] for column in capped_report["columns"]}

print(
    "capped:",
    capped_report["inference_capped"],
    "| scanned:",
    capped_report["inference_rows_scanned"],
)
print(
    "int_widens   full scan →", full_types["int_widens"], "| capped →", capped_types["int_widens"]
)
print(
    "leading_zero_id →",
    full_types["leading_zero_id"],
    "— zero-padded ids read as integers, so the padding is gone",
)

## 3. `extreme_types` — decimals at both ends

`decimal_hi` is `decimal128(24,21)` and round-trips through parquet exactly. `beyond_38`
carries 41 significant digits — past the decimal128 cap — so the CSV ladder **demotes it
to float64**. That demotion is documented behavior, not a defect; the tour shows the
precision it costs.

In [ ]:
extreme = datagen("extreme_types")
extreme_dir = extreme.write_files(rows=ROWS, seed=SEED)

typed = spark.read.parquet(str(extreme_dir / "data.parquet"))
typed_table = typed.orderBy("id").to_arrow()
print(typed_table.schema.field("decimal_hi").type)
print("parquet decimal_hi:", typed_table.to_pylist()[0]["decimal_hi"])
print("parquet beyond_38 :", typed_table.to_pylist()[0]["beyond_38"])

In [ ]:
csv_frame = spark.read.smartCsv(str(extreme_dir / "data.csv"), sep=",")
csv_types = {c["name"]: c["resolved_type"] for c in csv_frame.describe_ingest()["columns"]}
print(csv_types)

row = csv_frame.orderBy("id").to_arrow().to_pylist()[0]
print("csv decimal_hi:", row["decimal_hi"], f"({type(row['decimal_hi']).__name__})")
print("csv beyond_38 :", row["beyond_38"], f"({type(row['beyond_38']).__name__})")

## 4. `secrets` — credential-named columns read normally

Every value is obviously synthetic (`repark-fake-…`) and no value imitates a real
credential format. **Reads behave normally today**: nothing is masked, and nothing here
claims otherwise. Opt-in flagging of secret-shaped *data* columns is a roadmap feature
this fixture predates — today `prop_key_is_secret` governs configuration keys, not table
columns. `bucket_key` is the deliberate negative control (ends in `_key`, excluded by the
documented `bucket` carve-out).

In [ ]:
secrets = datagen("secrets")
secrets_dir = secrets.write_files(rows=ROWS, seed=SEED)

frame = spark.read.parquet(str(secrets_dir / "data.parquet"))
row = frame.orderBy("id").to_arrow().to_pylist()[0]
for column, class_id in secrets.SECRET_COLUMNS:
    print(f"{column:>14}  ({class_id:>16})  {row[column]}")

carve_out = secrets.CARVE_OUT_COLUMN
print(f"{carve_out:>14}  ({'negative control':>16})  {row[carve_out]}")

## 5. `smartcsv` — the messy-CSV zoo

One corpus per delimiter (`,` `;` tab `|`), each with a UTF-8 BOM, two preamble lines, a
duplicated header name, ragged rows in both directions, currency and euro-comma amounts,
eight bool spellings and seventeen null-token spellings.

The delimiter is **declared** below (`sep=`) rather than auto-detected — the next cell
shows why. European-locale `;` files should pass `sep=';'`.

Two reported findings live in this corpus, both pinned in
`python/repark/tests/test_datasets_facade.py`: auto-detect picks a rival delimiter, and
the euro-comma column (`760,35`) infers as `decimal128(5,2)` but **refuses loud** when the
cast meets the raw comma text. The value cell below therefore projects the columns it is
about; a whole-frame read of this corpus raises.

In [ ]:
smartcsv = datagen("smartcsv")
smart_dir = smartcsv.write_files(rows=ROWS, seed=SEED)

for scheme, delimiter in smartcsv.DELIMITERS.items():
    frame = spark.read.smartCsv(str(smart_dir / smartcsv.csv_file_name(scheme)), sep=delimiter)
    report = frame.describe_ingest()
    print(
        f"{scheme:>10}: delimiter={report['delimiter']!r} bom={report['bom_stripped']} "
        f"skipped={report['skipped_lines']} rows={report['data_row_count']} "
        f"ragged={report['ragged_rows_padded']} columns={len(report['columns'])}"
    )
print()
print("columns:", frame.columns)

In [ ]:
# Auto-detect scores candidates by field-count agreement, and `csv.reader` only honors a
# quote that STARTS a field. The `embedded_delims` value carries all four candidates, so a
# rival delimiter splits every line into two tidy fields and wins the vote. Declaring `sep`
# (European-locale: `sep=';'`) is the working path; the DS-4 pin records this as a
# known-limit, not a closed class.
from repark.spark._csv_smart import prepare_messy_csv

for scheme, delimiter in smartcsv.DELIMITERS.items():
    auto = prepare_messy_csv(smart_dir / smartcsv.csv_file_name(scheme))
    print(
        f"{scheme:>10}: wrote {delimiter!r} → auto-detected {auto.report.delimiter!r} "
        f"({auto.report.data_row_count} data rows instead of {ROWS})"
    )

In [ ]:
# Null tokens and bool spellings against the generator's typed truth.
frame = spark.read.smartCsv(str(smart_dir / smartcsv.csv_file_name("comma")), sep=",")
truth = smartcsv.small(rows=ROWS, seed=SEED).to_pylist()
table = (
    frame.select("id", "flag", "yes_no", "nullable_note", "amount_wide").orderBy("id").to_arrow()
)
rows = table.to_pylist()

print("flag:", table.schema.field("flag").type)
print("amount_wide:", table.schema.field("amount_wide").type, "=", rows[4]["amount_wide"])
# Rows 8-17 walk out of the recognized null spellings into the ones that stay literal.
for index in range(8, 18):
    print(
        f"{index}: note={rows[index]['nullable_note']!r} "
        f"(truth {truth[index]['nullable_note']!r}) "
        f"flag={rows[index]['flag']} yes_no={rows[index]['yes_no']!r}"
    )

## Where this is pinned

Nothing in a notebook is a gate. The behavior toured above is pinned, at CI scale, in:

* `python/repark/tests/test_datasets_facade.py` — the facade pins for all five families
  (Arrow path, value **and** type), including the `POLICY` and `BUG-CANDIDATE` markers
  for the sampling miss, the >38-digit demotion, the delimiter auto-detect
  known-limit (declare `sep=`; European-locale files use `sep=';'`), and the
  euro-comma cast refusal.
* `python/repark-parity/tests/test_datasets_*.py` — generator shape and determinism
  (pure pyarrow, no engine).
* `task/c18-datasets-ledger.md` — what each increment delivered and what it reported.

In [ ]:
spark.stop()